# AI-reviewed HTML coverage correction

User-authorized exploratory development. Same MPNet/base controls and seeds 17/42/73; a balanced 20-row training correction and 15 added validation pages. Labels are AI judgments. No final test is included. Preserve all artifacts before stopping the A100.

In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', WANDB_MODE='disabled', TOKENIZERS_PARALLELISM='false')
packages = ['torch==2.9.1','sentence-transformers==5.1.2','transformers==4.57.3','datasets==4.4.1','xgboost==3.1.2','scikit-learn==1.7.2','numpy==2.3.5','scipy==1.16.3','beautifulsoup4==4.14.2','accelerate==1.11.0']
for command in [[sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'], [sys.executable,'-m','pip','install','-q'] + packages]:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Environment setup failed')
print('Pinned environment installed. Model training uses fresh Python subprocesses.')


In [ ]:
import os, sys, json, hashlib, tarfile, subprocess
from pathlib import Path
output=Path('/content/mpnet_html_coverage_ai')
archive=Path('/content/html_coverage_ai_inputs.tar.gz')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='01b6d6730fe37359b73a0bb9e387c591252f2067662f29b7139e4d903f35545d'
assert not output.exists()
with tarfile.open(archive) as bundle: bundle.extractall('/content',filter='data')
protocol=json.loads((output/'protocol.json').read_text())
assert protocol['final_test_included'] is False and not (output/'holdout.jsonl').exists()
for name,expected in protocol['files'].items():
    with (output/name).open('rb') as stream: assert hashlib.file_digest(stream,'sha256').hexdigest()==expected,name

def run_stage(stage, command):
    with (output/(stage+'.log')).open('w') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=dict(os.environ,USE_TF='0',USE_FLAX='0',WANDB_MODE='disabled')) as process:
            for line in process.stdout:
                log.write(line);log.flush();print(line,end='',flush=True)
            if process.wait():raise RuntimeError(stage+' failed')
run_stage('base_integrity',[sys.executable,'-c',"import torch,hashlib,json; from huggingface_hub import hf_hub_download; assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(); p=hf_hub_download('sentence-transformers/all-mpnet-base-v2','model.safetensors',revision='e8c3b32edf5434bc2275fc9bab85f82640a19130'); h=hashlib.file_digest(open(p,'rb'),'sha256').hexdigest(); assert h=='78c0197b6159d92658e319bc1d72e4c73a9a03dd03815e70e555c5ef05615658'; print(json.dumps(dict(gpu=torch.cuda.get_device_name(),base_weight_sha256=h)))"])
print('FROZEN INPUTS VERIFIED',protocol['counts'],protocol['windows'])


In [ ]:
run_stage('train',[sys.executable,'-u',str(output/'supervised_pilot.py'),'train','--output',str(output)])


In [ ]:
for seed in [17,42,73]:
    folder=output/f'finetuned_{seed}'
    run_stage(f'export_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'export','--package',str(folder),'--assets',str(output/'inference_assets')])
    run_stage(f'replay_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'replay','--package',str(folder),'--input',str(output/'validation.jsonl'),'--expected',str(folder/'validation_predictions.jsonl'),'--output',str(folder/'package_replay.json')])
manifest={}
for path in output.rglob('*'):
    if path.is_file() and '__pycache__' not in path.parts and path.name not in ('artifact_manifest.json','completion.json'):
        with path.open('rb') as stream: digest=hashlib.file_digest(stream,'sha256').hexdigest()
        manifest[str(path.relative_to(output))]=dict(bytes=path.stat().st_size,sha256=digest)
(output/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
from datetime import datetime,timezone
completion=dict(completed_at=datetime.now(timezone.utc).isoformat(),models=6,portable_packages_replayed=3,validation_pages=207,final_test_evaluated=False,artifact_bytes=sum(r['bytes'] for r in manifest.values()),manifest_sha256=hashlib.sha256((output/'artifact_manifest.json').read_bytes()).hexdigest())
(output/'completion.json').write_text(json.dumps(completion,indent=2)+'\n')
print('HTML CORRECTION COMPLETED',json.dumps(completion))
